In [5]:
import re
import os
import logging as logger
import asyncio
from playwright.async_api import (
    Error,
    async_playwright,
    Playwright,
    BrowserContext,
    Page,
)
import time
import sys
from pathlib import Path
from typing import Optional, Callable, Dict, Any, List


# <----------------------------------------------------> #

BROWSER_INSTANCE = "chromium"
HEADLESS = False
USER_DATA_DIR = "../user_data"
BASE_URL = "https://web.whatsapp.com/"

In [6]:
async def initialize_playwright():
    # TODO: Perform browser level optimizations and other stuff
    playwright = await async_playwright().start()
    logger.info(f"Launching {BROWSER_INSTANCE} with persistent context...")

    browser = await playwright[BROWSER_INSTANCE].launch_persistent_context(
        USER_DATA_DIR, headless=HEADLESS
    )

    page = await browser.new_page()

    # await self.page_instance.set_viewport_size({"width": 1920, "height": 1080})
    logger.info(f"{BROWSER_INSTANCE} launched successfully.")
    return page


async def login(page):
    # TODO: QR code & phone number login via script
    if os.path.exists(USER_DATA_DIR):
        logger.info("User is already logged in. Skipping login step.")
    else:
        logger.info("User not logged in. Redirecting to WhatsApp login page.")

    await page.goto(BASE_URL)
    await page.bring_to_front()

    logger.info("Waiting for WhatsApp chats to load...")
    await page.wait_for_selector(
        '//*[@id="pane-side"]/div[2]/div/div/child::div', timeout=600000
    )
    logger.info("WhatsApp chats loaded.")

In [7]:
page = await initialize_playwright()
await login(page)

In [8]:
async def close_chat_panel(page):
    selector = "div._ajv7"
    span_selector = selector + "> div > span"
    if await page.locator(span_selector).get_attribute("aria-hidden") == "true":
        await page.locator(span_selector).click()

        await page.keyboard.press("ArrowDown")
        await page.keyboard.press("ArrowDown")
        await page.keyboard.press("ArrowDown")
        await page.keyboard.press("Enter")

In [ ]:
close_chat_panel(page)